# Bi-Mamba 32M — Chinese ↔ Vietnamese translation, end-to-end on Colab

Notebook chạy **toàn bộ pipeline** từ A đến Z: tải dữ liệu → train tokenizer → train Bi-Mamba → đánh giá SacreBLEU/chrF → dịch demo.

**Yêu cầu:** Runtime → GPU (T4 / L4 / A100 đều OK). RAM ≥ 12 GB.

> Sau bản sửa Mamba CUDA fast-path, hãy train một run mới sạch nếu checkpoint cũ được tạo trước fix. Lịch mặc định hiện là 60k steps, eval/save mỗi 1k steps, có early stopping.


## 1. Mount Google Drive (tuỳ chọn)

Để lưu checkpoint vĩnh viễn. Nếu không cần, bạn có thể bỏ qua ô này.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Skipping drive mount:', e)


## 2. Clone repo

Đang clone từ **`ChauDucToan/NLP_DHM`** — đổi `REPO_URL` nếu bạn fork sang chỗ khác.

In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
REPO_DIR = '/content/NLP_DHM'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only || true
%cd $REPO_DIR
!pwd && ls


## 3. Cài đặt dependencies cơ bản

Cài các package thuần Python (`torch`, `sentencepiece`, `sacrebleu`, ...). CUDA fast-path cho Mamba sẽ được cài ở ô **3b** ngay bên dưới.


In [ ]:
!pip install -q -r requirements.txt
import torch, sys
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
sys.path.insert(0, 'src')


### 3b. Tăng tốc Mamba bằng CUDA kernels (tuỳ chọn nhưng khuyên dùng)

Cài wheel **trực tiếp từ GitHub Releases** thay vì build từ PyPI (PyPI sẽ cố compile và hầu như fail trên Colab vì lệch ABI / CUDA toolkit). Mất ~30 giây. Train sẽ nhanh ~5–10× nếu thành công, fallback PyTorch vẫn chạy đúng nếu fail.


In [ ]:
# === Optional but recommended: CUDA fast-path (5–10× speedup) ===
# Repo code applies SiLU/gating outside the fused kernels so fast-path,
# reference forward, and step() keep the same semantics.
# `pip install mamba-ssm causal-conv1d` from PyPI tries to *build* from source
# on Colab and almost always fails (CUDA toolkit / torch ABI mismatch).
# Instead, install matching prebuilt wheels straight from the GitHub releases.
import sys, subprocess, torch

# Skip on CPU runtimes (no benefit, and the wheels need CUDA).
if not torch.cuda.is_available():
    print('No CUDA → skipping fast-path install (pure-PyTorch fallback will run).')
else:
    torch_mm  = '.'.join(torch.__version__.split('+')[0].split('.')[:2])  # e.g. '2.10'
    cuda_maj  = torch.version.cuda.split('.')[0]                          # e.g. '12'
    py_tag    = f'cp{sys.version_info.major}{sys.version_info.minor}'     # e.g. 'cp312'
    abi       = 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
    suffix    = f'cu{cuda_maj}torch{torch_mm}cxx11abi{abi}-{py_tag}-{py_tag}-linux_x86_64.whl'

    ccv1d_ver = '1.6.1'
    mamba_ver = '2.3.1'
    ccv1d_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{ccv1d_ver}.post4/causal_conv1d-{ccv1d_ver}+{suffix}'
    mamba_url = f'https://github.com/state-spaces/mamba/releases/download/v{mamba_ver}/mamba_ssm-{mamba_ver}+{suffix}'

    print('Installing causal-conv1d:', ccv1d_url.rsplit("/", 1)[-1])
    r1 = subprocess.run(['pip', 'install', '-q', '--no-deps', ccv1d_url])
    print('Installing mamba-ssm:    ', mamba_url.rsplit("/", 1)[-1])
    r2 = subprocess.run(['pip', 'install', '-q', '--no-deps', mamba_url])

    ok = True
    try:
        import causal_conv1d, mamba_ssm  # noqa
        from mamba_ssm.ops.selective_scan_interface import selective_scan_fn  # noqa
        from causal_conv1d import causal_conv1d_fn  # noqa
    except Exception as e:
        ok = False
        print('FAST-PATH INSTALL FAILED:', e)
        print(' → falling back to pure-PyTorch Mamba (training still works, ~5–10× slower).')
    if ok:
        print('CUDA fast-path active: selective_scan_fn + causal_conv1d_fn imported OK.')


## 4. Configuration

Sửa các giá trị bên dưới nếu muốn chạy nhanh / lâu hơn. Mặc định: subsample 200 K cặp, max_steps 30 K — phù hợp T4 trong 1.5–2 giờ.

In [ ]:
import os
import yaml
from pathlib import Path

cfg_path = Path('configs/bi_mamba_55m.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

# =====================================================================
# Data knobs
# =====================================================================
# `everyday` (v3, recommended default): TED2020 + WikiMatrix +
# OpenSubtitles (cap 20k) + NLLB confidence-filtered (LASER >= 1.10, cap 20k) +
# bible-uedin (cap 6k) -> ~135k clean pairs, bible only ~4% of mix.
# OpenCC Trad->Simp + Cantonese filter make zh side consistent Mandarin Simplified.
# Keep this preset while debugging model quality; increase data only after
# fast/reference/step behavior and BLEU evaluation are stable.
cfg['data']['preset']           = 'everyday'    # tiny | small | everyday | medium | large
cfg['data']['max_train_pairs']  = 400_000       # set to None to use everything
cfg['data']['max_valid_pairs']  = 5_000
cfg['data']['max_test_pairs']   = 5_000
# Strict per-language script + length-ratio filter (drops WikiMatrix noise).
cfg['data']['script_check']     = True
cfg['data']['min_zh_vi_ratio']  = 0.10
cfg['data']['max_zh_vi_ratio']  = 1.20
# Dialect / script normalisation. Without these flags the zh side is
# dialect-heterogeneous (TED2020 is mostly Cantonese/Traditional, bible-uedin
# Traditional, WikiMatrix mixed Trad/Simp/Classical).
cfg['data']['zh_normalize_simplified'] = True
cfg['data']['zh_filter_cantonese']     = True
# Per-source caps. Apply only if the source is in the preset.
cfg['data']['max_pairs_per_source'] = {
    'opensubtitles': 20_000,
    'wikimatrix':    100_000,
    'nllb':          20_000,
    'bible_uedin':   6_000,
}
# NLLB LASER threshold ablation candidates:
#   1.10 = more data, noisier; 1.20 = cleaner; 1.30 = very clean but small.
cfg['data']['min_score_per_source'] = {
    'nllb': 1.10,
}

# =====================================================================
# Tokenizer / model knobs (32.4M params, vocab 16k, d_model 384)
# =====================================================================
cfg['model']['vocab_size']      = 16_000
cfg['model']['d_model']         = 384
cfg['model']['d_ff']            = 960
cfg['tokenizer']['vocab_size']  = cfg['model']['vocab_size']
# Coverage 1.0 keeps every CJK char observed in training in the vocab.
cfg['tokenizer']['character_coverage'] = 1.0

# =====================================================================
# Train knobs after Mamba fast-path fix
# =====================================================================
# 120k steps was too long for the everyday corpus. Train a fresh run for
# 30k-60k steps, evaluate best.pt and best_ema.pt, then tune data/beam.
cfg['train']['max_steps']        = 60_000
cfg['train']['warmup_steps']     = 3_000
cfg['train']['lr']               = 3.0e-4
cfg['train']['min_lr']           = 1.0e-5
cfg['train']['weight_decay']     = 0.05
cfg['train']['label_smoothing']  = 0.08
cfg['train']['batch_size']       = 128          # 64 if you OOM; 256 on A100
cfg['train']['grad_accum_steps'] = 1
cfg['train']['amp_dtype']        = 'bf16'       # 'fp16' on old T4 if bf16 unsupported
cfg['train']['eval_every']       = 1_000
cfg['train']['save_every']       = 1_000
cfg['train']['log_every']        = 50

# Auto-detect CPU count.
_n_cpu = os.cpu_count() or 4
cfg['train']['num_workers'] = min(8, max(2, _n_cpu - 2))
print(f'Using num_workers={cfg["train"]["num_workers"]} (detected {_n_cpu} CPUs)')

# Length-bucketed batching — same VRAM, usually more tokens/sec.
cfg['train']['length_bucket']      = True
cfg['train']['bucket_pool_factor'] = 100
# EMA of weights for eval/inference.
cfg['train']['ema']            = True
cfg['train']['ema_decay']      = 0.9990
# Early stopping: 6 ticks × eval_every 1_000 = 6k stale steps -> stop.
cfg['train']['early_stopping_patience']   = 6
cfg['train']['early_stopping_min_delta']  = 0.0

# =====================================================================
# Eval knobs
# =====================================================================
cfg['eval']['beam_size']        = 4
cfg['eval']['length_penalty']   = {'zh2vi': 1.00, 'vi2zh': 0.90}
cfg['eval']['num_samples']      = 5_000

Path('configs/_colab.yaml').write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True)
)
print('Saved configs/_colab.yaml')


## 5. Tải + chuẩn bị dữ liệu (OPUS zh-vi)

`prepare_data.py` tải trực tiếp từ **OPUS** (`object.pouta.csc.fi`). Không cần Hugging Face dataset config — `Helsinki-NLP/opus-100` thực ra **không có** cặp zh-vi.

Presets:

| preset      | sources                                                                                                | ~ pairs   | download  |
|-------------|--------------------------------------------------------------------------------------------------------|-----------|-----------|
| `tiny`      | TED2020                                                                                                | 50 K      | 1 MB      |
| `small`     | TED2020 + WikiMatrix + bible-uedin (no caps)                                                           | 200 K     | 25 MB     |
| `everyday` *(default, v3)* | TED2020 + WikiMatrix + OpenSubtitles (20k) + NLLB (LASER ≥1.10, 20k) + bible-uedin (6k) | 135 K     | ~700 MB   |
| `medium`    | small + OpenSubtitles vi-zh_cn (uncapped, **NOT** recommended)                                         | 3 M       | 65 MB     |
| `large`     | medium + NLLB (cap 50k, no score filter — alignment-noisy)                                             | 30 M      | ~700 MB   |

Bạn cũng có thể truyền `--custom-jsonl path/to/your.jsonl` (mỗi dòng `{"zh": "...", "vi": "..."}`).

In [ ]:
!python scripts/prepare_data.py --config configs/_colab.yaml --preset {cfg['data']['preset']}


## 6. Train SentencePiece tokenizer (chia sẻ chung zh+vi)

In [ ]:
!python scripts/train_tokenizer.py --config configs/_colab.yaml


## 7. Khởi tạo model + kiểm tra số tham số

Mục tiêu thiết kế hiện tại: khoảng **32.4 M** tham số với `vocab_size=16k`, `d_model=384`, 5 encoder + 5 decoder layers.


In [ ]:
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.utils import count_parameters, human_format

model = BiMambaTranslator(ModelConfig(**cfg['model']))
n = count_parameters(model)
print(f'Bi-Mamba parameters: {human_format(n)}  ({n:,})')


## 8. Train end-to-end

Checkpoint sẽ được ghi vào `runs/bi_mamba_55m/`.

Sau khi sửa fast-path, checkpoint cũ không nên dùng để kết luận chất lượng cuối. Nếu run cũ được train trước bản fix, bắt đầu run mới sạch. Khi Colab ngắt giữa chừng, resume bằng `latest.pt` hoặc `checkpoint_stepN.pt`; các checkpoint này hiện lưu đủ model, optimizer, scaler, EMA, global step và early-stopping state. `latest_ema.pt` dùng cho evaluate/inference, không dùng làm checkpoint resume chính.


In [ ]:
# Uncomment this line when starting a clean run after the fast-path fix.
# !rm -rf runs/bi_mamba_55m

!python scripts/train.py --config configs/_colab.yaml


## 8b. Average + EMA của các checkpoint cuối

Sau train, so sánh `best.pt`, `best_ema.pt`, `latest.pt`, `latest_ema.pt` và checkpoint averaged. Với run ngắn hoặc bắt đầu overfit, `best_ema.pt` thường đáng tin hơn `latest.pt`; với run ổn định, `avg_last5_ema.pt` có thể nhỉnh hơn.

Không dùng checkpoint được tạo trước bản sửa fast-path để đánh giá chất lượng cuối.


In [ ]:
# Average last 5 model checkpoints, and last 5 EMA checkpoints, separately.
!python scripts/avg_ckpts.py --ckpts-dir runs/bi_mamba_55m --n 5
!python scripts/avg_ckpts.py --ckpts-dir runs/bi_mamba_55m --n 5 --ema
!ls -lh runs/bi_mamba_55m/*.pt | tail -10


## 9. Đánh giá SacreBLEU + chrF (cả hai chiều zh→vi và vi→zh)

Mỗi checkpoint được evaluate với `--length-buckets` để in thêm BLEU/chrF cho
bucket độ dài nguồn (`short: <20`, `medium: 20–50`, `long: ≥50` ký tự) —
giúp chẩn đoán model chết ở bucket nào. Beam ablation quét beam ∈ {2, 4, 6}
trên checkpoint mạnh nhất để so sánh.

In [ ]:
# Compare checkpoint variants on the test set (+ per-length-bucket breakdown).
import os
for label, ckpt in [
    ('latest',         'runs/bi_mamba_55m/latest.pt'),
    ('latest_ema',     'runs/bi_mamba_55m/latest_ema.pt'),
    ('best',           'runs/bi_mamba_55m/best.pt'),
    ('best_ema',       'runs/bi_mamba_55m/best_ema.pt'),
    ('avg_last5',      'runs/bi_mamba_55m/avg_last5.pt'),
    ('avg_last5_ema',  'runs/bi_mamba_55m/avg_last5_ema.pt'),
]:
    if not os.path.exists(ckpt):
        print(f'-- skip {label} (no file: {ckpt})'); continue
    print(f'\n=== {label} | beam=4 ===')
    !python scripts/evaluate.py --config configs/_colab.yaml --checkpoint {ckpt} --num-samples 5000 --beam-size 4 --length-buckets

# Beam ablation for the strongest likely checkpoint.
for ckpt in ['runs/bi_mamba_55m/best_ema.pt', 'runs/bi_mamba_55m/avg_last5_ema.pt']:
    if os.path.exists(ckpt):
        print(f'\n=== beam ablation: {ckpt} ===')
        for beam in [2, 4, 6]:
            print(f'\n--- beam={beam} ---')
            !python scripts/evaluate.py --config configs/_colab.yaml --checkpoint {ckpt} --num-samples 5000 --beam-size {beam}
        break

## 9b. Grid-sweep decoding (beam × length_penalty → CSV)

Tìm `(beam, length_penalty)` tối ưu cho từng chiều **mà không phải retrain**.
Script chạy grid độc lập hai chiều (`beam × lp_zh2vi` cho zh→vi,
`beam × lp_vi2zh` cho vi→zh — KHÔNG phải cartesian `lp_zh2vi × lp_vi2zh`
vì LP có semantics khác nhau).

Checkpoint load **một lần**; chỉ decoding params thay đổi mỗi cell.
CSV output có cột: `direction,beam,length_penalty,bucket,n,bleu,chrf`.

Grid mặc định 4 beam × 5 lp_zh2vi + 4 beam × 4 lp_vi2zh = **36 cells** trên
2000 câu → ~40–60 phút T4. Giảm `--num-samples 500` hoặc thu hẹp grid nếu muốn nhanh hơn.

In [ ]:
# Grid-sweep trên checkpoint mạnh nhất (ưu tiên avg_last5_ema > best_ema > best).
import os
candidates = [
    ('avg_last5_ema', 'runs/bi_mamba_55m/avg_last5_ema.pt'),
    ('best_ema',      'runs/bi_mamba_55m/best_ema.pt'),
    ('best',          'runs/bi_mamba_55m/best.pt'),
]
sweep_ckpt = next((p for _, p in candidates if os.path.exists(p)), None)

if sweep_ckpt is None:
    print('SKIP — không tìm thấy checkpoint nào.')
else:
    out_csv = f'runs/bi_mamba_55m/sweep_{os.path.basename(sweep_ckpt).replace(".pt","")}.csv'
    print(f'Sweep target: {sweep_ckpt}\nCSV output:   {out_csv}')
    !python scripts/sweep_decode.py --config configs/_colab.yaml --checkpoint {sweep_ckpt} --num-samples 2000 --beams 1 2 4 6 --lp-zh2vi 0.8 0.9 1.0 1.1 1.2 --lp-vi2zh 0.6 0.8 0.9 1.0 --out {out_csv}

    # Load CSV + show top-3 theo BLEU cho mỗi chiều.
    import pandas as pd
    df = pd.read_csv(out_csv)
    df_all = df[df['bucket'] == 'all']
    print('\n=== Top 3 cells theo BLEU ===')
    for d in ['zh2vi', 'vi2zh']:
        top = df_all[df_all['direction'] == d].nlargest(3, 'bleu')
        print(f'\n[{d}]')
        print(top[['beam', 'length_penalty', 'bleu', 'chrf', 'n']].to_string(index=False))

## 10. Demo dịch

In [ ]:
import os
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.tokenizer import Tokenizer
from bi_mamba_mt.translate import translate_batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_candidates = [
    f"{cfg['train']['output_dir']}/avg_last5_ema.pt",
    f"{cfg['train']['output_dir']}/best_ema.pt",
    f"{cfg['train']['output_dir']}/best.pt",
    f"{cfg['train']['output_dir']}/latest.pt",
]
ckpt_path = next(p for p in _candidates if os.path.exists(p))
print('Loading', ckpt_path)
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model = BiMambaTranslator(ModelConfig(**ckpt['model_cfg'])).to(device)
model.load_state_dict(ckpt['model'], strict=False); model.eval()
tok = Tokenizer(f"{cfg['data']['tokenizer_dir']}/spm.model")

zh_samples = [
    '你好，世界。',
    '今天的天气很好。',
    '这本书我已经读完了。',
]
vi_samples = [
    'Xin chào thế giới.',
    'Hôm nay trời thật đẹp.',
    'Tôi đã đọc xong cuốn sách này.',
]

beam = int(cfg['eval']['beam_size'])
lp = cfg['eval']['length_penalty']

print('--- zh → vi ---')
for s, t in zip(zh_samples,
                translate_batch(model, tok, zh_samples, 'zh2vi',
                                beam_size=beam, length_penalty=lp['zh2vi'], device=device)):
    print(f'  {s!r} -> {t!r}')

print('--- vi → zh ---')
for s, t in zip(vi_samples,
                translate_batch(model, tok, vi_samples, 'vi2zh',
                                beam_size=beam, length_penalty=lp['vi2zh'], device=device)):
    print(f'  {s!r} -> {t!r}')


## 11. (Tuỳ chọn) Lưu checkpoint vào Drive

In [ ]:
import shutil, os
DRIVE_DIR = '/content/drive/MyDrive/bi_mamba_55m'
if os.path.isdir('/content/drive/MyDrive'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for fname in ('latest.pt', 'final.pt', 'config.yaml'):
        src = f"{cfg['train']['output_dir']}/{fname}"
        if os.path.isfile(src):
            shutil.copy(src, DRIVE_DIR)
            print('Copied', src, '->', DRIVE_DIR)
    tok_dst = f'{DRIVE_DIR}/tokenizer'
    os.makedirs(tok_dst, exist_ok=True)
    for fname in ('spm.model', 'spm.vocab'):
        src = f"{cfg['data']['tokenizer_dir']}/{fname}"
        if os.path.isfile(src):
            shutil.copy(src, tok_dst)
    print('Done. Files in', DRIVE_DIR)
else:
    print('Drive not mounted; skipping.')
